# DS602 Midterm

Fake Reviews Dataset.

https://osf.io/3vds7

Goal is to identify the `label`. 

* Use random_state = 120
* Don't use any pretrained ML model/library
* If you need to encode the target variable (label), you may use [LabelEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html)

In [1]:
import pandas as pd
# you can import additional libraries

In [2]:
df = pd.read_csv('https://osf.io/download/3vds7/')

In [4]:
df.head()

,category,rating,label,text_
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...


Variables in the Dataset

category: Represents the type of product being reviewed. It has been encoded numerically to facilitate processing by the machine learning model.

rating: This is the star rating given by the reviewer, ranging from 1 to 5. Ratings are a common feature in review data and can be influential in determining the authenticity of a review.

text_: The actual text of the review. This is the primary data used for analyzing and classifying the reviews as genuine or fake.

label: Indicates whether the review is genuine or fake. This is the target variable for your model.


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40432 entries, 0 to 40431
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   category  40432 non-null  object 
 1   rating    40432 non-null  float64
 2   label     40432 non-null  object 
 3   text_     40432 non-null  object 
dtypes: float64(1), object(3)
memory usage: 1.2+ MB


In [6]:
import seaborn as sns

In [7]:
df.rating.value_counts()

5.0    24559
4.0     7965
3.0     3786
1.0     2155
2.0     1967
Name: rating, dtype: int64

In [8]:
df.category.unique()

array(['Home_and_Kitchen_5', 'Sports_and_Outdoors_5', 'Electronics_5',
       'Movies_and_TV_5', 'Tools_and_Home_Improvement_5',
       'Pet_Supplies_5', 'Kindle_Store_5', 'Books_5', 'Toys_and_Games_5',
       'Clothing_Shoes_and_Jewelry_5'], dtype=object)

In [9]:
df.label.unique()

array(['CG', 'OR'], dtype=object)

In [10]:
df.columns

Index(['category', 'rating', 'label', 'text_'], dtype='object')

In [35]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder



# Encode the 'label' column
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['label'])
df['category'] = label_encoder.fit_transform(df['category'])
# Split the data into training and testing sets
X = df[['category', 'rating', 'text_']]  # Features
y = df['label']  # Target variable

# Data Cleaning and Type Conversion
X['category'] = pd.to_numeric(X['category'], errors='coerce')
X['rating'] = pd.to_numeric(X['rating'], errors='coerce')

# Handling Missing Values (if needed)
#X = X.dropna()  # Remove rows with missing values



X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=120)

# Feature extraction (TF-IDF for the 'text_' column)
tfidf_vectorizer = TfidfVectorizer()
X_train_text_tfidf = tfidf_vectorizer.fit_transform(X_train['text_'])
X_test_text_tfidf = tfidf_vectorizer.transform(X_test['text_'])

# Combine the TF-IDF features with 'category' and 'rating' columns
import scipy.sparse
from scipy.sparse import hstack

X_train_category_rating = X_train[['category', 'rating']]
X_test_category_rating = X_test[['category', 'rating']]

X_train_combined = hstack((X_train_text_tfidf, X_train_category_rating))
X_test_combined = hstack((X_test_text_tfidf, X_test_category_rating))

# Choose a classification algorithm (e.g., Naive Bayes)
classifier = MultinomialNB()
classifier.fit(X_train_combined, y_train)

# Make predictions on the test set
y_pred = classifier.predict(X_test_combined)

# Evaluate the model's performance
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print("  ")
# Generate a classification report
print(classification_report(y_test, y_pred))


C:\Users\Admin\AppData\Local\Temp\ipykernel_6808\4245669556.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['category'] = pd.to_numeric(X['category'], errors='coerce')
C:\Users\Admin\AppData\Local\Temp\ipykernel_6808\4245669556.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['rating'] = pd.to_numeric(X['rating'], errors='coerce')


Accuracy: 0.8402374180783975
  
              precision    recall  f1-score   support

           0       0.78      0.95      0.85      3982
           1       0.94      0.73      0.82      4105

    accuracy                           0.84      8087
   macro avg       0.86      0.84      0.84      8087
weighted avg       0.86      0.84      0.84      8087



In [23]:
X.shape

(40432, 3)

In [24]:
y.shape

(40432,)